# Do your motifs have names?

**Question this notebook answers:** a motif is a recurring cluster of actions
that co-occur more often than chance. Whether a given motif is worth anything
is decided by reading the commands behind it, not by its score.

This notebook extracts the motifs from your graph, then traces each one back
to real commands (masked for secrets) so you can classify them yourself:

| Category | What it looks like | What to do |
|---|---|---|
| **Usable** | A cluster that names a real workflow step | A candidate for procedural memory |
| **Trivial** | Real co-occurrence with no semantic value (`cd` + `ls`) | Ignore it |
| **Artefact** | Commands sharing atoms but unrelated in intent | Community detection merged unrelated things |

**Run notebook 02 first** to find the right granularity for your corpus.
Then paste the `url_segments` and `path_components` values in the
configuration cell below.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SESSIONS_GLOB = "~/.openclaw/agents/*/sessions/*.jsonl"

# Paste the granularity settings from notebook 02 here.
GRANULARITY = {
    "url_segments": 3,
    "path_components": 2,
}

# Optional: filter to a specific agent name (leave empty to include all).
AGENT_FILTER = ""

# Minimum number of hyperedges a pair of atoms must appear in together
# before it is counted as an edge.
MIN_OCCURRENCES = 2

# Minimum number of atoms in a community for it to be reported as a motif.
MIN_SIZE = 3
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import glob
import os

from agentgraph import build_hyperedges, metrics, motifs

matched = glob.glob(os.path.expanduser(SESSIONS_GLOB))
if not matched:
    raise FileNotFoundError(
        f"No session files found for pattern: {SESSIONS_GLOB!r}\n"
        "Update SESSIONS_GLOB in the configuration cell above."
    )
print(f"{len(matched)} session file(s) found.")

In [ ]:
# Build hyperedges at the chosen granularity.
edges = build_hyperedges(SESSIONS_GLOB, **GRANULARITY)

if AGENT_FILTER:
    edges = [e for e in edges if e.agent == AGENT_FILTER]
    print(f"Filtered to agent '{AGENT_FILTER}': {len(edges)} hyperedges.")
else:
    print(f"Hyperedges: {len(edges)}")

print()
print(metrics(edges, MIN_OCCURRENCES))

In [ ]:
# Extract motifs.
results = motifs(edges, min_occurrences=MIN_OCCURRENCES, min_size=MIN_SIZE)

if not results:
    print("No motifs found with the current settings.")
    print("Try lowering MIN_SIZE or MIN_OCCURRENCES, or collecting more traces.")
else:
    print(f"{len(results)} motif(s) found.")

In [ ]:
# Display each motif with its atoms and masked example commands.
# Classify each motif yourself: usable / trivial / artefact.

for i, motif in enumerate(results, 1):
    print(f"{'=' * 70}")
    print(f"Motif #{i}  |  {motif['size']} atoms  |  {motif['occurrences']} matching commands")
    print(f"{'─' * 70}")
    print("Atoms:")
    for atom in motif["atoms"]:
        print(f"  {atom}")
    print()
    print("Example commands (masked):")
    if motif["examples"]:
        for j, cmd in enumerate(motif["examples"], 1):
            print(f"  [{j}] {cmd[:160]}")
    else:
        print("  (no examples — all commands were filtered by dedup or masking)")
    print()
    print("  → Your classification: [ usable / trivial / artefact ]")
    print()

In [ ]:
# Cross-reference: find the real commands behind any atom of interest.
# Set ATOM_QUERY to any substring of an atom name to see its commands.

ATOM_QUERY = "git"  # Change this to any atom fragment

from agentgraph import mask_secrets

matching = [
    e for e in edges
    if any(ATOM_QUERY in atom for atom in e.atoms)
]

print(f"{len(matching)} hyperedge(s) contain an atom matching '{ATOM_QUERY}'.")
print()
for e in matching[:5]:
    print(f"Atoms   : {e.atoms}")
    print(f"Command : {mask_secrets(e.command)[:200]}")
    print()

## How to classify motifs

**Usable** — you can name what the motif does without looking at the code.
Examples: "deploy pipeline" (build + push + restart), "code review cycle"
(checkout + diff + commit). These are candidates for procedural memory.

**Trivial** — statistically real, semantically empty. `cd` then `ls` is the
standard case: every navigation step produces it. A high count of trivial
motifs says the corpus has little to extract; it does not mean the parameters
are wrong.

**Artefact** — the atoms come from commands that happen to share a binary or a
path but describe unrelated workflows. Community detection merged them on a
few edge cases, and reading the example commands shows it.

Extracting preconditions, sequencing steps and storing procedures is worth
doing on usable motifs only.

Continue to **notebook 04** to measure whether they repeat over time.